In [0]:
# ==================================================
# NOTEBOOK 02: DATA CLEANING & FILTERING
# ==================================================

print("="* 80)
print("Notebook 02: Data cleaning & filtering")
print("="*80)

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import warnings
import builtins
warnings.filterwarnings('ignore')

print("Libraries imported successfully")
print()

In [0]:
# =================================================
#  Step 1: Load Raw Accepted Loans from Notebook 01
# =================================================

print("Step 01: Loading Raw Accepted Loans from Notebook 01")
print("="*80)

accepted_path = "/Volumes/workspace/lending_club/lending_club_data/accepted_2007_to_2018Q4.csv"

accepted_df = spark.read.csv(accepted_path,header = True, inferSchema = True)

row_count = accepted_df.count()
print(f"Raw accepeted loans loaded: {row_count} rows,{len(accepted_df.columns)} columns ")


In [0]:
 # ===============================================
 # Step 2: Filter to 2015-2018
 # ===============================================

 print("Step 02: Filtering to 2015-2018")
 print("="*80)

 before_count = accepted_df.count()

 filtered_df = (accepted_df
                .withColumn('issue_date',expr("try_to_date(issue_d,'MMM-yyyy')"))
                .filter(col('issue_date').isNotNull())
                .filter(year(col('issue_date')).between(2015,2018))
 )

after_count = filtered_df.count()
removed_rows = before_count - after_count

print(f"Before filter: {before_count} rows")
print(f"After filter: {after_count} rows")
print(f"Rows removed: {removed_rows} rows")
print("Years retained: 2015,2016,2017,2018")
print()

# Confirm year distribution
print("Row count by year after filtering")
year_df = (filtered_df.groupBy(year(col('issue_date')).alias('Year'))\
    .count()\
        .orderBy(col('Year'))
        .withColumn('Year',col('Year').cast(StringType()))
)
            
# Total row
total_df = filtered_df.agg(
    lit('Total').alias('Year'),
    count('*').alias('count'))

year_df.union(total_df).show()


In [0]:
# =====================================
# Step 3: Drop Columns with >50% Null values
# =====================================

print("Step 3: Dropping High Null Columns (>50% nulls)")
print("="*80)

total_rows = filtered_df.count()

null_count_rows = (
    filtered_df.select([
        sum(col(c).isNull().cast("int")).alias(c)
        for c in filtered_df.columns
    ]).collect()[0]
)

high_null_cols = [
    c for c in filtered_df.columns 
    if (null_count_rows[c]/total_rows * 100) > 50
]

print(f"Columns with >50% nulls: {len(high_null_cols)}")  
print(f"Columns to drop: {high_null_cols}")
print()

# Drop high null columns
df_dropped = filtered_df.drop(*high_null_cols)

print(f"Columns before drop: {len(filtered_df.columns)}")
print(f"Columns after drop: {len(df_dropped.columns)}")
print(f"Columns removed: {len(high_null_cols)}")


In [0]:
# ====================================
# Step 3.5: Examine Data types Before Fixing
# ====================================

print("Step 3.5: Examing Data types Before fixing")
print("="* 80)

# All column data types
print("All column data types:")
print()
dtypes_list  = df_dropped.dtypes

# Get unique data types from all columns
print("unique data types:")
unique_dtypes = set(dtype for _,dtype in dtypes_list)
print(unique_dtypes)
print()

string_cols = [col for col,dtype in dtypes_list if dtype == 'string']
date_cols = [col for col,dtype in dtypes_list if dtype == 'date']
double_cols = [col for col,dtype in dtypes_list if dtype == 'double' ]

print(f"Columns with string data type [{len(string_cols)}]: {string_cols} ")
print()
print(f"Columns with date data type [{len(date_cols)}]: {date_cols}")
print()
print(f"Columns with double data type [{len(double_cols)}]: {double_cols}")
print()


In [0]:
display(df_dropped.select(string_cols))

In [0]:
# ======================================================================
# Auto-detect Numeric String Columns
# ======================================================================

print("Auto-detecting numeric string columns...")
print("=" * 80)

# Sample 1000 rows to check each string column
sample_df = df_dropped.sample(fraction=0.001, seed=42)

numeric_string_cols = []
non_numeric_string_cols = []

for c in string_cols:
    try:
        # Try casting to Double — if most values succeed, it's numeric
        total = sample_df.count()
        success = sample_df.filter(
            col(c).cast(DoubleType()).isNotNull() | col(c).isNull()
        ).count()
        
        success_rate = success / total * 100
        
        if success_rate >= 95:  # 95% or more values cast successfully
            numeric_string_cols.append(c)
        else:
            non_numeric_string_cols.append(c)
    except:
        non_numeric_string_cols.append(c)

print(f"String columns that should be Double [{len(numeric_string_cols)}]:")
print(numeric_string_cols)
print()
print(f"String columns that are genuinely text [{len(non_numeric_string_cols)}]:")
print(non_numeric_string_cols)

In [0]:
# ======================================================================
# STEP 4: Fix Data Types
# ======================================================================

print("STEP 4: Fixing Data Types")
print("=" * 80)

exclude_from_casting = ['id', 'member_id', 'url', 'zip_code']

# Re-run auto casting with exclusions
df_typed = df_dropped

# 1. Auto-cast all numeric string columns to Double
print("Casting numeric string columns to Double:")
for c in numeric_string_cols:
    if c not in exclude_from_casting:
        df_typed = df_typed.withColumn(c, col(c).cast(DoubleType()))
    print(f"  {c} → Double")

print()

# 2. Clean emp_length — extract numeric value
if 'emp_length' in df_typed.columns:
    df_typed = df_typed.withColumn(
        'emp_length',
        regexp_replace(col('emp_length'), '[^0-9]', '').cast(IntegerType())
    )
    print("emp_length — extracted numeric, cast to Integer")

# 3. Clean term — extract numeric value
if 'term' in df_typed.columns:
    df_typed = df_typed.withColumn(
        'term',
        regexp_replace(col('term'), '[^0-9]', '').cast(IntegerType())
    )
    print("term — extracted numeric, cast to Integer")

# 4. Parse date string columns
date_cols_to_fix = ['earliest_cr_line', 'last_pymnt_d',
                    'last_credit_pull_d', 'next_pymnt_d']

for c in date_cols_to_fix:
    if c in df_typed.columns:
        df_typed = df_typed.withColumn(
            c, expr(f"try_to_date({c}, 'MMM-yyyy')")
        )
        print(f"{c} — parsed to Date")

print()

# 6. Cast id back to String — it is an identifier, not a numeric value
if 'id' in df_typed.columns:
    df_typed = df_typed.withColumn('id', col('id').cast(StringType()))
    print("✓ id — cast back to String (identifier column)")

# 7. Confirm results
print("Data types after fixing:")
new_dtypes = dict(df_typed.dtypes)
new_string_cols = [c for c, t in new_dtypes.items() if t == 'string']
new_double_cols = [c for c, t in new_dtypes.items() if t == 'double']
new_date_cols   = [c for c, t in new_dtypes.items() if t == 'date']
new_int_cols    = [c for c, t in new_dtypes.items() if 'int' in t.lower()]

print(f"String columns : {len(new_string_cols)} → {new_string_cols}")
print(f"Double columns : {len(new_double_cols)}")
print(f"Date columns   : {len(new_date_cols)}")
print(f"Integer columns: {len(new_int_cols)}")

In [0]:
# ======================================================================
# Drop unnecessary columns
# ======================================================================

cols_to_drop = [
    'url',        # just a web link, no analytical value
    'title',      # free text, redundant with purpose column
    'zip_code',   # partially masked e.g. 190xx — unusable
    'pymnt_plan', # nearly all values are 'n' — no variation
    'member_id',  # not needed — id is the unique identifier
]

# Only drop if they exist
existing_drop_cols = [c for c in cols_to_drop if c in df_typed.columns]
df_typed = df_typed.drop(*existing_drop_cols)

print(f"Dropped {len(existing_drop_cols)} unnecessary columns: {existing_drop_cols}")
print(f"Columns remaining: {len(df_typed.columns)}")

In [0]:
display(df_typed)